# Regressione logistica

In [13]:
import pandas as pd
import numpy as np


from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

from joblib import Parallel, delayed
import math
import re
from tabulate import tabulate
from pathlib import Path
import warnings 
from sklearn.exceptions import ConvergenceWarning
import time
from datetime import timedelta
# Nascondo i warning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [14]:
def training(file_path, csv_name):
      
    df = pd.read_csv(file_path)

    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']
    df_validi = df.dropna(subset=original_target_list).copy()

    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)

    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID'] 

    # Sostituisco infiniti con NaN prima di calcolare la media
    features.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Riempo i NaN con la media
    features = features.fillna(features.mean())
    
    # Fisso un range per non andare in overflow
    features = features.clip(lower=-1e10, upper=1e10)

    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    cv = GroupKFold(n_splits=5)

    
    logistic_pipeline = Pipeline([
        ('scaler', RobustScaler()), 
        ('classifier', LogisticRegression(
            random_state=42, 
            n_jobs=1,               
            class_weight='balanced',              
            solver='saga',         
            max_iter=2000          
        ))
    ])
    
    multi_output_model = MultiOutputClassifier(logistic_pipeline)

    iperparametri = {
        'estimator__classifier__C': [0.01, 0.1, 1], 
        'estimator__classifier__penalty': ['l2', 'elasticnet'], 
        'estimator__classifier__l1_ratio': [0.5],               
    }

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")

    # Aggiungo un gestore di warning specifico per l'overflow durante il fit
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'overflow encountered')
        warnings.filterwarnings('ignore', 'invalid value encountered')
        
        grid_search = GridSearchCV(
            estimator=multi_output_model,
            param_grid=iperparametri,
            cv=cv,                  
            scoring=scorer,         
            n_jobs=-1,              
            verbose=1,              
            refit=True,             
            error_score=0.0 # Se esplode un fit, restituisce 0 invece di crashare tutto
        )

        grid_search.fit(features, target, groups=groups)

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    fold_reports = []

    clean_best_params = {k.replace('estimator__classifier__', ''): v for k, v in best_params.items()}
    
    final_params = {
        'classifier__random_state': 42,
        'classifier__n_jobs': 1,
        'classifier__class_weight': 'balanced',
        'classifier__solver': 'saga',
        'classifier__max_iter': 2000,
        **{f'classifier__{k}': v for k, v in clean_best_params.items()}
    }

    for train_idx, test_idx in cv.split(features, target, groups):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Ricostruisco pipeline clone
        base_pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('classifier', LogisticRegression()) 
        ])
        
        base_pipeline.set_params(**final_params)
        
        model_clone = MultiOutputClassifier(base_pipeline)
        model_clone.fit(X_train, y_train)
        
        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)

        report_dict = {}
        for i, col in enumerate(final_target_list):
            rep = classification_report(
                y_test.iloc[:, i],
                y_pred[:, i],
                output_dict=True,
                zero_division=0
            )
            
            unique_classes = np.unique(y_test.iloc[:, i])

            if len(unique_classes) < 2:
                auc_val = np.nan 
            else:
                try:
                    if y_proba_list[i].shape[1] == 2:
                        auc_val = roc_auc_score(y_test.iloc[:, i], y_proba_list[i][:, 1])
                    else:
                        auc_val = 0.5 
                except ValueError:
                    auc_val = np.nan
            
            rep['auc'] = auc_val
            report_dict[col] = rep
        fold_reports.append(report_dict)

    final_result = [{
        **clean_best_params,
        'mean_score': best_score, 
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports 
    }]

    return final_result


# Stampo i risultati in un formato leggibile

In [ ]:
def print_grid_search_results(results_per_dataset):
    
    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI (Logistic Regression)")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        best_result = metrics_list[0]

        auc_values = []
        if best_result.get('fold_reports'):
            for fold_rep in best_result['fold_reports']:
                for target_key, target_metrics in fold_rep.items():
                    if isinstance(target_metrics, dict) and 'auc' in target_metrics:
                        auc_values.append(target_metrics['auc'])
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            mean_auc = np.nanmean(auc_values) if auc_values else 0.0
        
        if np.isnan(mean_auc): mean_auc = 0.0
        

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}")
        print(f" Mean AUC    = {mean_auc:.3f}\n")

        print("Iperparametri Ottimali:")
        
        possible_params = [
            ('C (Inverse Reg)', 'C'),       
            ('Penalty', 'penalty'),         
            ('L1 Ratio', 'l1_ratio'),       
            ('Solver', 'solver')            
        ]
        
        params_table = []
        for label, key in possible_params:
            val = best_result.get(key)
            if val is not None:
                 params_table.append([label, val])

        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))

        print("\n Metriche di Classificazione per Target (Dettaglio primo fold):\n")
        
        target_names = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

        if best_result.get('fold_reports'):
            first_fold_report = best_result['fold_reports'][0]

            for target_name in target_names:
                if target_name not in first_fold_report:
                    continue

                current_target_report = first_fold_report[target_name]

                rows = []
                classes = [c for c in ['0', '1'] if c in current_target_report]

                for cls in classes:
                    rows.append([
                        f"Classe {cls}",
                        f"{current_target_report[cls]['precision']:.3f}",
                        f"{current_target_report[cls]['recall']:.3f}",
                        f"{current_target_report[cls]['f1-score']:.3f}",
                        int(current_target_report[cls]['support'])
                    ])

                # AUC per singolo target
                auc_val = current_target_report.get('auc')
                if auc_val is not None and not np.isnan(auc_val):
                    auc_str = f"  ---> AUC: {auc_val:.3f}"
                else:
                    auc_str = "  ---> AUC: N/A"

                print(f"  Target: {target_name} {auc_str}")
                print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'],
                             tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
                print()
        else:
            print("Nessun report dettagliato disponibile.")

        # Raccolta dati per il riepilogo finale
        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            f"{mean_auc:.3f}",  # Aggiungo la AUC
            best_result.get('C'),
            best_result.get('penalty'),
            best_result.get('l1_ratio') if best_result.get('penalty') == 'elasticnet' else '-'
        ])


    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'AUC', 'C', 'Penalty', 'L1 Ratio'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.3f', '.0f', '.0f', '.0f')))

# Eseguo il tutto

In [16]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)



end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search (6 combinazioni) per: t2_medsam
Fitting 5 folds for each of 6 candidates, totalling 30 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when


Inizio Grid Search (6 combinazioni) per: t2_preprocessed
Fitting 5 folds for each of 6 candidates, totalling 30 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)


Inizio Grid Search (6 combinazioni) per: t2_original
Fitting 5 folds for each of 6 candidates, totalling 30 fits


...


Inizio Grid Search (6 combinazioni) per: preprocessed_dynamic
Fitting 5 folds for each of 6 candidates, totalling 30 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when


Inizio Grid Search (6 combinazioni) per: original_dynamic
Fitting 5 folds for each of 6 candidates, totalling 30 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when


                         RIEPILOGO DEI MIGLIORI RISULTATI (Logistic Regression)

────────────────────────────────────────────────────────────────────────────────
 Dataset: t2_medsam
────────────────────────────────────────────────────────────────────────────────

 Performance: F1-score = 0.582 ± 0.072
 Mean AUC    = 0.623

Iperparametri Ottimali:
Parametro        Valore
---------------  ----------
C (Inverse Reg)  1
Penalty          elasticnet
L1 Ratio         0.5

 Metriche di Classificazione per Target (Dettaglio primo fold):

  Target: PR_class   ---> AUC: 0.593
           Precision    Recall    F1-score    Support
--------  -----------  --------  ----------  ---------
Classe 0     0.167      0.333      0.222         3
Classe 1     0.667      0.444      0.533         9

  Target: ER_class   ---> AUC: 0.727
           Precision    Recall    F1-score    Support
--------  -----------  --------  ----------  ---------
Classe 0      0.2         1        0.333         1
Classe 1       1  